# splicetarget — Quickstart Guide

End-to-end long-read splice isoform analysis and ASO target nomination.

This notebook demonstrates how to run the full `splicetarget` pipeline using the **Python API**.  
For CLI usage, see the [README](../README.md).

---

## Prerequisites

```bash
pip install -e ".[dev]"          # or: conda env create -f env.yml
```

**External tools** (for alignment stage only): `minimap2 >= 2.26`, `samtools >= 1.18`

---

## 1 — Imports

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from splicetarget.data.io import ReadRecord
from splicetarget.data.reference import Exon, Gene, Transcript
from splicetarget.isoforms.collapse import collapse_isoforms
from splicetarget.isoforms.classify import classify_isoforms, IsoformCategory
from splicetarget.isoforms.quantify import quantify_isoforms
from splicetarget.splicing.events import detect_aberrant_events
from splicetarget.therapeutic.aso_design import ASOCandidate, ASOStrategy
from splicetarget.therapeutic.scoring import ScoringWeights, rescore_candidates, candidates_to_dataframe
from splicetarget.utils.genome import gc_content, reverse_complement

plt.rcParams.update({
    "figure.dpi": 140, "font.size": 10,
    "font.family": "sans-serif",
    "axes.spines.top": False, "axes.spines.right": False,
})

PALETTE = {
    "FSM": "#27AE60", "ISM": "#2ECC71", "NIC": "#F39C12",
    "NNC": "#E74C3C", "IR": "#E67E22", "GG": "#95A5A6",
}

print("splicetarget loaded ✓")

---
## 2 — Define the Reference Gene Model

In production you will load this from a GTF/GFF3 annotation via `ReferenceTranscriptome`.  
Here we construct a 5-exon gene by hand for demonstration.

```
Exon 1       Exon 2       Exon 3       Exon 4       Exon 5
▓▓▓▓▓▓▓▓─────▓▓▓▓▓▓───────▓▓▓▓▓▓▓▓▓────▓▓▓▓▓────────▓▓▓▓▓▓▓▓▓▓▓▓
1000  1200   2000 2150    3000  3300    4000 4100     5000     5500
```

In [ ]:
tx = Transcript(
    transcript_id="TX001", gene_id="GENE001", gene_name="CandidateGene",
    chrom="chr1", start=1000, end=5500, strand="+",
    biotype="protein_coding",
    exons=[
        Exon(chrom="chr1", start=1000, end=1200, strand="+", exon_number=1),
        Exon(chrom="chr1", start=2000, end=2150, strand="+", exon_number=2),
        Exon(chrom="chr1", start=3000, end=3300, strand="+", exon_number=3),
        Exon(chrom="chr1", start=4000, end=4100, strand="+", exon_number=4),
        Exon(chrom="chr1", start=5000, end=5500, strand="+", exon_number=5),
    ],
)
gene = Gene(
    gene_id="GENE001", gene_name="CandidateGene",
    chrom="chr1", start=1000, end=5500, strand="+",
    biotype="protein_coding", transcripts={"TX001": tx},
)
print(f"{gene.gene_name}  {gene.chrom}:{gene.start:,}-{gene.end:,}  ({len(tx.exons)} exons)")

---
## 3 — Simulate Patient Long Reads

Four read populations mimicking a patient with mixed aberrant splicing.  
In production, these come from `iter_aligned_reads()` on a BAM file.

| Population | Reads | Pattern |
|------------|------:|----------------------------------------|
| Normal (FSM) | 1,623 | All 5 exons |
| Exon skip | 53 | Exon 2 missing |
| Cryptic exon | 489 | Novel exon at 2500–2600 |
| Intron retention | 322 | Exon 2 fused through intron into exon 3 |

In [ ]:
def _reads(prefix, n, blocks):
    return [ReadRecord(name=f"{prefix}_{i}", sequence="A"*500, quality=None,
                       chrom="chr1", start=blocks[0][0], end=blocks[-1][1],
                       strand="+", is_mapped=True, exon_blocks=blocks)
            for i in range(n)]

all_reads = (
    _reads("normal", 1623, [(1000,1200),(2000,2150),(3000,3300),(4000,4100),(5000,5500)])
  + _reads("skip",     53, [(1000,1200),(3000,3300),(4000,4100),(5000,5500)])
  + _reads("cryptic", 489, [(1000,1200),(2000,2150),(2500,2600),(3000,3300),(4000,4100),(5000,5500)])
  + _reads("ir",      322, [(1000,1200),(2000,3300),(4000,4100),(5000,5500)])
)
print(f"{len(all_reads):,} reads generated")

---
## 4 — Run the Pipeline

The four core stages run in sequence.  
Each returns structured Python objects that feed into the next stage.

In [ ]:
# Stage 1 — Collapse reads into unique isoforms
isoforms = collapse_isoforms(all_reads, junction_tolerance=10,
                              end_tolerance=50, min_reads=1,
                              gene_name="CandidateGene")

# Stage 2 — Classify isoforms against reference (SQANTI3-style)
classified = classify_isoforms(isoforms, gene, junction_tolerance=10)

# Stage 3 — Quantify expression
expression = quantify_isoforms(classified)

# Stage 4 — Detect aberrant splicing events
events = detect_aberrant_events(classified, gene)

print(f"Isoforms: {len(isoforms)}  |  Events: {len(events)}  |  "
      f"Therapy candidates: {sum(1 for c in classified if c.is_therapy_candidate)}")

---
## 5 — Inspect Results

In [ ]:
# ── Isoform classification table ──
pd.DataFrame([{
    "Isoform": c.isoform.isoform_id,
    "Category": c.category.value,
    "Reads": f"{c.isoform.read_count:,}",
    "Expr %": f"{next((e.fraction for e in expression if e.isoform_id == c.isoform.isoform_id), 0):.1%}",
    "Therapy?": "✓" if c.is_therapy_candidate else "",
} for c in classified])

In [ ]:
# ── Aberrant events ──
if events:
    display(pd.DataFrame([{
        "Event": ev.event_id,
        "Type": ev.event_type.value,
        "Region": f"{ev.chrom}:{ev.start:,}-{ev.end:,}",
        "Reads": ev.total_read_support,
        "Frame impact": ev.reading_frame_impact,
        "ASO target?": "✓" if ev.is_aso_target else "",
    } for ev in events]))
else:
    print("No aberrant events detected.")

---
## 6 — Visualize

In [ ]:
# ── Dual panel: classification pie + expression bar ──
fig, (ax_pie, ax_bar) = plt.subplots(1, 2, figsize=(13, 4.5),
                                      gridspec_kw={"width_ratios": [1, 1.4]})

# Pie (weighted by read count)
cat_reads = {}
for c in classified:
    cat_reads[c.category.value] = cat_reads.get(c.category.value, 0) + c.isoform.read_count
labels, sizes, colors = zip(*[
    (k, v, PALETTE.get(k, "#BDC3C7")) for k, v in cat_reads.items()
])
ax_pie.pie(sizes, labels=labels, colors=colors,
           autopct=lambda p: f"{p:.1f}%" if p > 3 else "",
           startangle=140, wedgeprops={"edgecolor": "white", "linewidth": 1.5})
ax_pie.set_title("Isoform Classification (by reads)", fontweight="bold")

# Bar
expr_sorted = sorted(expression, key=lambda e: e.fraction)
ax_bar.barh([e.isoform_id for e in expr_sorted],
            [e.fraction * 100 for e in expr_sorted],
            color=[PALETTE.get(e.category, "#BDC3C7") for e in expr_sorted],
            edgecolor="white", height=0.65)
ax_bar.set_xlabel("% of Gene Expression")
ax_bar.set_title("Expression Profile", fontweight="bold")
for i, e in enumerate(expr_sorted):
    ax_bar.text(e.fraction * 100 + 0.4, i, f"{e.read_count:,}",
               va="center", fontsize=8, color="#555")
plt.tight_layout()
plt.show()

In [ ]:
# ── Sashimi-style gene model ──
n_tracks = len(classified) + 2
fig, axes = plt.subplots(n_tracks, 1, figsize=(14, 1.5 * n_tracks), sharex=True,
                          gridspec_kw={"hspace": 0.06,
                                       "height_ratios": [1.3] + [1]*len(classified) + [0.5]})
xlo, xhi = gene.start - 400, gene.end + 400

def _draw(ax, blocks, col, lbl, y=0.5, h=0.55):
    for j, (s, e) in enumerate(blocks):
        ax.add_patch(plt.Rectangle((s, y-h/2), e-s, h, fc=col, ec="k", lw=.6, zorder=3))
        if j:
            xs = np.linspace(blocks[j-1][1], s, 30)
            ax.plot(xs, y + .35*np.sin(np.linspace(0, np.pi, 30)),
                    color="#95A5A6", lw=1, zorder=2)
    ax.set_xlim(xlo, xhi); ax.set_ylim(-.1, 1.15); ax.set_yticks([])
    ax.set_ylabel(lbl, rotation=0, ha="right", va="center", fontsize=8.5, fontweight="bold")
    for sp in ax.spines.values(): sp.set_visible(False)

# Reference
_draw(axes[0], [(ex.start, ex.end) for ex in tx.exons], "#2C3E50", "REF", h=.7)
axes[0].set_title(f"  {gene.gene_name}    {gene.chrom}:{gene.start:,}–{gene.end:,}",
                  fontsize=12, fontweight="bold", loc="left", color="#2C3E50")
for ex in tx.exons:
    axes[0].text((ex.start+ex.end)/2, .02, f"E{ex.exon_number}",
                 ha="center", va="bottom", fontsize=7, color="#7F8C8D")

# Patient isoforms
for i, c in enumerate(classified):
    col = PALETTE.get(c.category.value, "#95A5A6")
    _draw(axes[i+1], c.isoform.exon_blocks, col, f"{c.category.value}\n{c.isoform.read_count:,}×")
    if c.category == IsoformCategory.IR and c.retained_introns:
        for rs, re_ in c.retained_introns:
            axes[i+1].axvspan(rs, re_, alpha=.12, color="#E67E22", zorder=1)
            axes[i+1].text((rs+re_)/2, 1, "▼ IR", ha="center", fontsize=7,
                           color="#E67E22", fontweight="bold")

# Legend
axes[-1].axis("off")
axes[-2].set_xlabel("Genomic position (bp)", fontsize=9)
axes[-2].spines["bottom"].set_visible(True)
plt.tight_layout()
plt.show()

---
## 7 — ASO Candidate Scoring (demo)

The full `design_aso_candidates()` function requires an indexed FASTA to extract target sequences.  
Here we demonstrate the scoring module with manually constructed candidates.

In [ ]:
candidates = []
for i, (off, seq) in enumerate([
    (0,  "AGCTTAGCGCAATTCCGGA"),
    (5,  "GCAATTCCGGACTTCAGCTA"),
    (10, "CCGGACTTCAGCTAACGTACG"),
    (15, "ACTTCAGCTAACGTACGCCAA"),
    (2,  "CTTAGCGCAATTCCGGACT"),
]):
    candidates.append(ASOCandidate(
        candidate_id=f"ASO_{i+1:03d}", event_id="ES_001",
        strategy=ASOStrategy.EXON_INCLUSION,
        chrom="chr1", target_start=2000+off, target_end=2000+off+len(seq),
        strand="+", target_sequence=seq, aso_sequence=reverse_complement(seq),
        gc_score=.7+.05*i, self_comp_score=.8-.1*i,
        accessibility_score=.6+.08*i, splice_proximity_score=.9-.15*i,
        length_score=.75, aso_length=len(seq), gc_content=gc_content(seq),
        distance_to_splice_site=off, target_region_type="exonic",
        off_target_hits=i,
    ))

ranked = rescore_candidates(candidates, ScoringWeights())
candidates_to_dataframe(ranked)

In [ ]:
# ── Score breakdown ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 3.5),
                                gridspec_kw={"width_ratios": [1, 1.6]})
ids = [c.candidate_id for c in ranked]
sc = [c.composite_score for c in ranked]
ax1.barh(ids, sc, color=["#1ABC9C" if s >= .6 else "#E0E0E0" for s in sc],
         edgecolor="white", height=.6)
ax1.axvline(.6, color="#E74C3C", ls="--", lw=1.2, label="Threshold")
ax1.set_xlabel("Composite Score"); ax1.set_title("Ranking", fontweight="bold")
ax1.invert_yaxis(); ax1.legend(fontsize=8)

comps = {
    "GC":       [c.gc_score*.20 for c in ranked],
    "Self-comp":[c.self_comp_score*.15 for c in ranked],
    "Proximity":[c.splice_proximity_score*.20 for c in ranked],
    "Length":   [c.length_score*.10 for c in ranked],
    "Access":   [c.accessibility_score*.15 for c in ranked],
    "Off-tgt":  [max(0,1-c.off_target_hits/5)*.20 for c in ranked],
}
cols2 = ["#3498DB","#9B59B6","#E74C3C","#F39C12","#1ABC9C","#2C3E50"]
left = np.zeros(len(ranked))
for (nm, vs), co in zip(comps.items(), cols2):
    ax2.barh(ids, vs, left=left, color=co, edgecolor="white", height=.6, label=nm)
    left += np.array(vs)
ax2.set_xlabel("Weighted Contribution")
ax2.set_title("Score Breakdown", fontweight="bold")
ax2.invert_yaxis(); ax2.legend(fontsize=7, ncol=3, loc="lower right")
plt.tight_layout(); plt.show()

---
## 8 — Clinical Summary

In [ ]:
from IPython.display import Markdown

total = sum(c.isoform.read_count for c in classified)
ab = [c for c in classified if c.category.is_aberrant]
ab_pct = sum(c.isoform.read_count for c in ab) / total * 100
hc = sum(1 for c in ranked if c.is_high_confidence)

Markdown(f"""
### Clinical Summary

| Metric | Value |
|--------|-------|
| **Gene** | {gene.gene_name} ({gene.chrom}:{gene.start:,}–{gene.end:,}) |
| **Total reads** | {total:,} |
| **Unique isoforms** | {len(classified)} |
| **Known (FSM/ISM)** | {sum(1 for c in classified if c.category.is_known)} |
| **Novel (NIC/NNC/IR)** | {sum(1 for c in classified if c.category.is_novel)} |
| **Aberrant expression** | {ab_pct:.1f}% |
| **Events detected** | {len(events)} |
| **Therapy candidates** | {sum(1 for c in classified if c.is_therapy_candidate)} |
| **High-confidence ASOs** | {hc} (score ≥ 0.6, off-targets ≤ 3) |

**Interpretation:** {ab_pct:.0f}% of gene expression derives from aberrant splice forms.
{len(events)} actionable events were detected. {hc} ASO candidates met the
high-confidence threshold for experimental validation.
""")

---

## Running on Real Data

Replace the synthetic reads above with real BAM input:

```python
from splicetarget.data.io import iter_aligned_reads
from splicetarget.data.reference import ReferenceTranscriptome

ref = ReferenceTranscriptome("gencode.v44.annotation.gtf")
gene = ref.get_gene("DMD")

reads = list(iter_aligned_reads(
    "patient_isoseq.bam",
    region=f"{gene.chrom}:{gene.start}-{gene.end}",
    min_mapq=20,
    require_splice=True,
))
```

Or use the CLI:

```bash
splicetarget run \
    --reads patient_isoseq.bam \
    --reference GRCh38.fa \
    --annotation gencode.v44.annotation.gtf \
    --gene DMD \
    --outdir results/
```